# TPE Optimizer (Optuna)

Joint-portfolio TPE search. See `docs/research.md` for methodology.

## Setup

In [1]:
# ruff: noqa: E402
import sys
from pathlib import Path

for _candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_candidate / "prediction_market_extensions").is_dir() and (
        _candidate / "backtests"
    ).is_dir():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

from prediction_market_extensions.backtesting._notebook_support import ensure_notebook_repo_context

repo_root = ensure_notebook_repo_context()

## Configuration

In [2]:
from decimal import Decimal

from prediction_market_extensions.backtesting._execution_config import ExecutionModelConfig
from prediction_market_extensions.backtesting._execution_config import StaticLatencyConfig
from prediction_market_extensions.backtesting._experiments import ParameterSearchExperiment
from prediction_market_extensions.backtesting._prediction_market_runner import MarketDataConfig
from prediction_market_extensions.backtesting._replay_specs import QuoteReplay
from prediction_market_extensions.backtesting.data_sources import PMXT, Polymarket, QuoteTick
from prediction_market_extensions.backtesting.optimizers import ParameterSearchConfig
from prediction_market_extensions.backtesting.optimizers import ParameterSearchWindow


NAME = "notebook_tpe_optimizer"

DATA = MarketDataConfig(
    platform=Polymarket,
    data_type=QuoteTick,
    vendor=PMXT,
    sources=(
        "local:/Volumes/LaCie/pmxt_raws",
        "archive:r2.pmxt.dev",
        "relay:209-209-10-83.sslip.io",
    ),
)

# Joint-portfolio: every trial evaluates ONE parameter set across all 5 replays
# simultaneously. PnL and fills sum, drawdown is computed on the summed equity
# curve, coverage is averaged across markets.
#
# All 5 slugs are long-dated example markets. PMXT quote-tick collection
# started in late February 2026, so every window below stays on/after
# 2026-03-01 — otherwise the optimizer rejects the trial with
# invalid_result_count.
BASE_REPLAYS = (
    QuoteReplay(market_slug="human-moon-landing-in-2026", token_index=0),
    QuoteReplay(market_slug="new-coronavirus-pandemic-in-2026", token_index=0),
    QuoteReplay(
        market_slug="will-openais-market-cap-be-between-750b-and-1t-at-market-close-on-ipo-day",
        token_index=0,
    ),
    QuoteReplay(market_slug="okx-ipo-in-2026", token_index=0),
    QuoteReplay(market_slug="nothing-ever-happens-2026", token_index=0),
)

TRAIN_WINDOWS = (
    ParameterSearchWindow(
        name="train-early-march",
        start_time="2026-03-01T00:00:00Z",
        end_time="2026-03-15T23:59:59Z",
    ),
    ParameterSearchWindow(
        name="train-late-march",
        start_time="2026-03-16T00:00:00Z",
        end_time="2026-03-31T23:59:59Z",
    ),
    ParameterSearchWindow(
        name="train-early-april",
        start_time="2026-04-01T00:00:00Z",
        end_time="2026-04-05T23:59:59Z",
    ),
)

HOLDOUT_WINDOWS = (
    ParameterSearchWindow(
        name="holdout-april",
        start_time="2026-04-06T00:00:00Z",
        end_time="2026-04-11T23:59:59Z",
    ),
)

STRATEGY_SPEC = {
    "strategy_path": "strategies:QuoteTickEMACrossoverStrategy",
    "config_path": "strategies:QuoteTickEMACrossoverConfig",
    "config": {
        "trade_size": Decimal("5"),
        "fast_period": "__SEARCH__:fast_period",
        "slow_period": "__SEARCH__:slow_period",
        "entry_buffer": "__SEARCH__:entry_buffer",
        "take_profit": "__SEARCH__:take_profit",
        "stop_loss": "__SEARCH__:stop_loss",
    },
}

# Continuous / log-scale space — TPE samples within these bounds.
PARAMETER_SPACE = {
    "fast_period": {"type": "int", "low": 16, "high": 128},
    "slow_period": {"type": "int", "low": 96, "high": 512},
    "entry_buffer": {"type": "float", "low": 1e-4, "high": 2e-3, "log": True},
    "take_profit": {"type": "float", "low": 2e-3, "high": 3e-2, "log": True},
    "stop_loss": {"type": "float", "low": 2e-3, "high": 3e-2, "log": True},
}

EXECUTION = ExecutionModelConfig(
    queue_position=True,
    latency_model=StaticLatencyConfig(
        base_latency_ms=75.0,
        insert_latency_ms=10.0,
        update_latency_ms=5.0,
        cancel_latency_ms=5.0,
    ),
)

# Smoke-test sizing: keep trials low to validate the pipeline end-to-end
# (slugs load, windows align, scores are finite, holdout replay completes)
# before scaling. TPE's advantage over random kicks in around ~20 trials;
# bump MAX_TRIALS to 50–100 once the leaderboard shows real non-sentinel
# scores.
MAX_TRIALS = 12
HOLDOUT_TOP_K = 3

PARAMETER_SEARCH = ParameterSearchConfig(
    name=NAME,
    data=DATA,
    base_replays=BASE_REPLAYS,
    strategy_spec=STRATEGY_SPEC,
    parameter_space=PARAMETER_SPACE,
    train_windows=TRAIN_WINDOWS,
    holdout_windows=HOLDOUT_WINDOWS,
    max_trials=MAX_TRIALS,
    random_seed=7,
    holdout_top_k=HOLDOUT_TOP_K,
    initial_cash=100.0,
    probability_window=256,
    min_quotes=500,
    min_price_range=0.005,
    min_fills_per_window=1,
    execution=EXECUTION,
    emit_html=False,
    chart_output_path="output",
    sampler="tpe",
)

EXPERIMENT = ParameterSearchExperiment(
    name=NAME,
    description="EMA TPE joint-portfolio search on PMXT L2 data",
    parameter_search=PARAMETER_SEARCH,
)

print(
    f"Configured TPE search: {MAX_TRIALS} trials over "
    f"{len(BASE_REPLAYS)} markets (joint portfolio), "
    f"{len(TRAIN_WINDOWS)} train window(s), {len(HOLDOUT_WINDOWS)} holdout window(s)."
)

Configured TPE search: 12 trials over 5 markets (joint portfolio), 3 train window(s), 1 holdout window(s).


## Run optimizer

In [3]:
from pprint import pprint

from prediction_market_extensions.backtesting._experiments import run_experiment_async
from prediction_market_extensions.backtesting._notebook_support import suppress_notebook_cell_output

with suppress_notebook_cell_output():
    summary = await run_experiment_async(EXPERIMENT)

print("Selected params:")
pprint(dict(summary.selected_params))
print("Leaderboard CSV:", summary.leaderboard_csv_path)
print("Summary JSON:", summary.summary_json_path)

Selected params:
{'entry_buffer': 0.0015110674897952736,
 'fast_period': 97,
 'slow_period': 268,
 'stop_loss': 0.01488173010872023,
 'take_profit': 0.003260300027864126}
Leaderboard CSV: /Users/evankolberg/prediction-market-backtesting/output/notebook_tpe_optimizer_leaderboard.csv
Summary JSON: /Users/evankolberg/prediction-market-backtesting/output/notebook_tpe_optimizer_summary.json


## Leaderboard

In [4]:
import pandas as pd
from IPython.display import display, Markdown

_leaderboard_df = pd.read_csv(summary.leaderboard_csv_path)
_display_cols = [
    c
    for c in (
        "trial_id",
        "train_median_score",
        "holdout_median_score",
        "train_median_pnl",
        "holdout_median_pnl",
        "train_median_drawdown",
        "train_median_fills",
        "params_json",
    )
    if c in _leaderboard_df.columns
]
_top = _leaderboard_df[_display_cols].head(10).copy()

_fmt = {
    "train_median_score": "{:.4f}".format,
    "holdout_median_score": "{:.4f}".format,
    "train_median_pnl": "{:.4f}".format,
    "holdout_median_pnl": "{:.4f}".format,
    "train_median_drawdown": "{:.4f}".format,
    "train_median_fills": "{:.1f}".format,
}
_fmt = {k: v for k, v in _fmt.items() if k in _top.columns}

_styled = (
    _top.style.format(_fmt, na_rep="-")
    .hide(axis="index")
    .set_caption(f"Top candidates — {EXPERIMENT.name} (TPE)")
)
display(_styled)

display(
    Markdown(
        "### Selected parameters\n\n"
        + "\n".join(f"- **{k}**: `{v}`" for k, v in dict(summary.selected_params).items())
    )
)

trial_id,train_median_score,holdout_median_score,train_median_pnl,holdout_median_pnl,train_median_drawdown,train_median_fills,params_json
10,-149.5251,-148.4974,-99.6205,-98.9791,99.6834,651.0,"{""entry_buffer"": 0.0015110674897952736, ""fast_period"": 97, ""slow_period"": 268, ""stop_loss"": 0.01488173010872023, ""take_profit"": 0.003260300027864126}"
9,-149.6747,-149.6218,-99.7831,-99.7429,99.7831,652.0,"{""entry_buffer"": 0.000716632368394708, ""fast_period"": 67, ""slow_period"": 243, ""stop_loss"": 0.006933702984315349, ""take_profit"": 0.00545251246568588}"
12,-149.6549,-744.6891,-99.7699,-98.8432,99.7699,636.0,"{""entry_buffer"": 0.0008411099653765187, ""fast_period"": 58, ""slow_period"": 222, ""stop_loss"": 0.013491002868039713, ""take_profit"": 0.004322780542319501}"
6,-149.6998,-,-99.7999,-,99.7999,669.0,"{""entry_buffer"": 0.0009469035303373606, ""fast_period"": 31, ""slow_period"": 314, ""stop_loss"": 0.0070982296398728, ""take_profit"": 0.012241950538095943}"
5,-149.7343,-,-99.7185,-,99.7185,650.0,"{""entry_buffer"": 0.00019935675399995848, ""fast_period"": 83, ""slow_period"": 492, ""stop_loss"": 0.02345567474483301, ""take_profit"": 0.00883291793197674}"
2,-149.7364,-,-99.8190,-,99.8243,689.0,"{""entry_buffer"": 0.0001240911146748937, ""fast_period"": 76, ""slow_period"": 304, ""stop_loss"": 0.0077435023669868795, ""take_profit"": 0.00413750692578901}"
3,-149.7379,-,-99.8236,-,99.8286,690.0,"{""entry_buffer"": 0.00031305153729655524, ""fast_period"": 92, ""slow_period"": 431, ""stop_loss"": 0.004364309247234048, ""take_profit"": 0.00239098668116823}"
1,-149.7535,-,-99.8357,-,99.8357,618.0,"{""entry_buffer"": 0.0003718635067633045, ""fast_period"": 24, ""slow_period"": 421, ""stop_loss"": 0.02826408380687946, ""take_profit"": 0.014187016294426116}"
7,-149.7747,-,-99.8035,-,99.8485,709.0,"{""entry_buffer"": 0.0003051291159941079, ""fast_period"": 39, ""slow_period"": 300, ""stop_loss"": 0.005387043471412535, ""take_profit"": 0.007286136798672624}"
8,-554.9290,-,-99.7179,-,99.8505,705.0,"{""entry_buffer"": 0.0002561630547236078, ""fast_period"": 110, ""slow_period"": 416, ""stop_loss"": 0.004223659200317334, ""take_profit"": 0.009429521261776949}"


### Selected parameters

- **fast_period**: `97`
- **slow_period**: `268`
- **entry_buffer**: `0.0015110674897952736`
- **take_profit**: `0.003260300027864126`
- **stop_loss**: `0.01488173010872023`

## Combined holdout chart

In [5]:
import gc

from prediction_market_extensions.backtesting._notebook_support import (
    display_html_artifacts,
    find_updated_html_artifacts,
    snapshot_html_artifacts,
    suppress_notebook_cell_output,
)
from prediction_market_extensions.backtesting._prediction_market_backtest import MarketReportConfig
from prediction_market_extensions.backtesting.optimizers import (
    build_parameter_search_window_backtest,
)
from prediction_market_extensions.backtesting.prediction_market import finalize_market_results

selected_window = HOLDOUT_WINDOWS[0] if HOLDOUT_WINDOWS else TRAIN_WINDOWS[-1]

output_root = repo_root / "output"
html_snapshot = snapshot_html_artifacts(output_root)

COMBINED_PLOT_PANELS = (
    "total_equity",
    "equity",
    "market_pnl",
    "periodic_pnl",
    "yes_price",
    "allocation",
    "total_drawdown",
    "drawdown",
    "total_rolling_sharpe",
    "rolling_sharpe",
    "total_cash_equity",
    "cash_equity",
    "monthly_returns",
    "total_brier_advantage",
    "brier_advantage",
)

COMBINED_REPORT = MarketReportConfig(
    count_key="quotes",
    count_label="Quotes",
    pnl_label="PnL (USDC)",
    market_key="slug",
    summary_report=True,
    summary_report_path=f"output/{NAME}_holdout_joint_portfolio.html",
    summary_plot_panels=COMBINED_PLOT_PANELS,
)

backtest = build_parameter_search_window_backtest(
    config=PARAMETER_SEARCH,
    window=selected_window,
    params=summary.selected_params,
    trial_id=0,
    name=f"{NAME}:{selected_window.name}:selected-params",
    emit_html=False,
    chart_output_path=str(output_root),
    return_summary_series=True,
)

with suppress_notebook_cell_output():
    results = await backtest.run_async()
    finalize_market_results(
        name=backtest.name,
        results=results,
        report=COMBINED_REPORT,
        multi_replay_mode="joint_portfolio",
    )

updated_html = find_updated_html_artifacts(output_root, html_snapshot)
num_markets = len(results)
print(f"Replayed {num_markets} joint-portfolio market(s) on window '{selected_window.name}'.")
display_html_artifacts(updated_html, repo_root=repo_root)

# Holdout replay stays in joint-portfolio mode. The summary chart overlays
# market-level series in the non-total panels and keeps portfolio totals in the
# total panels.
del results, backtest, updated_html, html_snapshot
gc.collect()


Replayed 5 joint-portfolio market(s) on window 'holdout-april'.


**Chart artifact:** `output/notebook_tpe_optimizer_holdout_joint_portfolio.html`

/Users/evankolberg/prediction-market-backtesting/.venv/lib/python3.13/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


28196